Notebook illustrating how the results of the paper were obtained. We only consider data from ILL11 for 2018. 

Data can be downloaded from: https://doi.org/10.12686/SED/NETWORKS/XP

In [ ]:
# import obspy
import os
import numpy as np
import pandas as pd

from joblib import dump, load

# from datamod.preproc_utils import preproc_stream
from isolation_forest.utils import train_if, compute_scores

First we train an isolation forest to the miniseed files contained in the folder "../data/XP/2018/ILL11/HHZ.D".

Note that the suffixes contains the day of the year, which we use to sort the files.

In [25]:
folder = "../data/XP/2018/ILL11/HHZ.D/"
stream_paths = [folder + f for f in os.listdir(folder) if not f.startswith("._")]
stream_paths = np.array(sorted(stream_paths, key=lambda f: int(f.rsplit(".", 1)[-1])))

Now we train the isolation forest. This will take a couple of minutes.

In [26]:
try:
    if_mod = load("../output/if_models/ILL11.joblib")

except (FileNotFoundError, OSError):
    if_mod = train_if(stream_paths)
    dump(if_mod, "../output/if_models/ILL11.joblib")

Next we run the trained isolation forest over the training and/or testing period and extract time series of the anomaly scores.

Note. The miniseed files we worked with correspond to a given day. When we extract sliding windows, there can be a small amount of the waveform towards the end of the day that is not accessed, because it is not sufficiently long to form a full window. In such cases the waveform of the next day (if available) is appended to this residual waveform.

The function below takes a few minutes to run.


In [29]:
try:
    scores_df = pd.read_csv("../output/scores/ILL11_anomaly_scores.csv", index_col=0)

except (FileNotFoundError, OSError):
    scores_df = compute_scores(stream_paths, if_mod)
    scores_df.to_csv("../output/scores/ILL11_anomaly_scores.csv")